# 🧠 Tutorial 2: Physics-Constrained ML Surrogates & Virtual Soft Sensors

In this tutorial, you will:
1. Load and benchmark pretrained Machine Learning yield surrogate models.
2. Validate physics constraints (elemental conservation & energy balances).
3. Extract raw hardware sensor telemetry with simulated sensor noise.
4. Evaluate the **6 Inferential Virtual Soft Sensors** with Bayesian 95% Uncertainty Intervals.

In [ ]:
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.simulation.plant_simulator import BiomassPlantSimulator
from src.sensors.telemetry import TelemetryExtractor
from src.sensors.soft_sensor_engine import SoftSensorSuite
from src.sensors.calibration import SoftSensorCalibrator

print("[*] Modules imported.")

## 1. Comparing First-Principles vs ML Surrogate Predictions

In [ ]:
simulator = BiomassPlantSimulator()

det_report = simulator.run_simulation(
    feedstock_name="pine_sawdust",
    feed_rate_kg_h=100.0,
    reactor_temp_c=520.0,
    yield_mode="DETERMINISTIC"
)

ml_report = simulator.run_simulation(
    feedstock_name="pine_sawdust",
    feed_rate_kg_h=100.0,
    reactor_temp_c=520.0,
    yield_mode="ML_SURROGATE"
)

print(f"Deterministic Yields -> Bio-Oil: {det_report.reactor.yields_dry.bio_oil_yield*100:.2f}%, Biochar: {det_report.reactor.yields_dry.biochar_yield*100:.2f}%, Syngas: {det_report.reactor.yields_dry.syngas_yield*100:.2f}%")
print(f"ML Surrogate Yields  -> Bio-Oil: {ml_report.reactor.yields_dry.bio_oil_yield*100:.2f}%, Biochar: {ml_report.reactor.yields_dry.biochar_yield*100:.2f}%, Syngas: {ml_report.reactor.yields_dry.syngas_yield*100:.2f}%")

## 2. Virtual Soft Sensors with 95% Bayesian Confidence Intervals

In [ ]:
telemetry = TelemetryExtractor.extract_from_report(ml_report, add_sensor_noise=True)

chk = ROOT_DIR / "models" / "checkpoints" / "soft_sensors.joblib"
if not chk.is_file():
    SoftSensorCalibrator().calibrate()
suite = SoftSensorSuite.load(chk)

estimates = suite.estimate_all(telemetry)

print("=== 6 Inferential Soft Sensor Predictions (95% UQ) ===")
for tag, est in estimates.items():
    print(f"[{tag}] {est.name:<32}: {est.point_estimate:>6.2f} {est.unit:<8} (95% CI: [{est.lower_95_ci:>6.2f} - {est.upper_95_ci:>6.2f}]) -> Status: {est.status}")